# Notebook 10 — COCO Annotation Audit and YOLO Conversion

| Field | Value |
|---|---|
| **Task ID** | DET-COCO-001 |
| **Phase** | 8 — COCO Conversion |
| **Owner** | Member 4 (Detection owner) |
| **Date** | 2026-09-21 |
| **Dataset** | COCO Car Damage Detection Dataset |
| **Structure** | `train/` (59 images), `val/` (11 images), `test/` (8 images) |
| **Annotations** | `COCO_*_annos.json` (damage), `COCO_mul_*_annos.json` (parts) |
| **Environment** | Google Colab / Local |

## Purpose

Prove that the COCO car-damage detection annotations are correct **before** any model training begins.
This notebook audits the annotations and converts them to YOLO TXT format for two distinct models:
1. **Generic Damage Detection** (`yolo_damage/`): Single-class detector (`damage`) locating damaged regions.
2. **Damaged Part Detection** (`yolo_parts/`): 5-class detector (`headlamp`, `front_bumper`, `hood`, `door`, `rear_bumper`).

## Hypothesis

> The COCO annotations are geometrically valid and the YOLO conversion preserves all bounding box positions within a round-trip tolerance of 1e-6.

In [ ]:
# Cell 2 — Environment detection, path configuration, versions
import os
import sys
import importlib
from pathlib import Path

SEED = 42

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')

    COLAB_BASE = Path('/content/NPN-Car-Insurance')
    if not COLAB_BASE.exists():
        COLAB_BASE = Path('/content')
    REPO_ROOT = COLAB_BASE
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / 'ml').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / 'ml' / 'src'
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

# Candidate paths for locating train/val/test folders
candidate_paths = [
    REPO_ROOT / 'data' / 'raw' / 'coco_car_damage' / 'coco-car-damage-detection-dataset' / 'coco-car-damage-detection-dataset',
    REPO_ROOT / 'data' / 'raw' / 'coco_car_damage' / 'coco-car-damage-detection-dataset',
    REPO_ROOT / 'data' / 'raw' / 'coco_car_damage',
    REPO_ROOT / 'data' / 'raw',
    Path('/content/NPN-Car-Insurance/data/raw/coco_car_damage/coco-car-damage-detection-dataset/coco-car-damage-detection-dataset'),
    Path('/content/data/raw/coco_car_damage/coco-car-damage-detection-dataset/coco-car-damage-detection-dataset'),
]

RAW_DIR = None
for p in candidate_paths:
    if (p / 'train').exists() and (p / 'val').exists():
        RAW_DIR = p
        break

if RAW_DIR is None:
    RAW_DIR = candidate_paths[0]

if IN_COLAB:
    os.system('pip install pyyaml opencv-python-headless matplotlib -q')

print(f'Environment   : {"Google Colab" if IN_COLAB else "Local"}')
print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'RAW_DIR       : {RAW_DIR}')
print(f'RAW_DIR exists: {RAW_DIR.exists()}')
print(f'SEED          : {SEED}')

import cv2
import numpy as np
import yaml

print('\n--- Package versions ---')
for pkg in ['cv2', 'numpy', 'yaml', 'matplotlib']:
    try:
        mod = importlib.import_module(pkg)
        print(f'  {pkg:<15} {getattr(mod, "__version__", "unknown")}')
    except ImportError:
        print(f'  {pkg:<15} NOT INSTALLED')

In [ ]:
# Cell 3 — Load all COCO JSONs for Damage and Parts
import json
from claimvision_ml.detection.coco_converter import COCOtoYOLOConverter

SPLITS = ['train', 'val', 'test']
coco_damage = {}
coco_parts = {}

damage_anno_filenames = {
    'train': 'COCO_train_annos.json',
    'val': 'COCO_val_annos.json',
    'test': 'COCO_test_annos.json',
}

parts_anno_filenames = {
    'train': 'COCO_mul_train_annos.json',
    'val': 'COCO_mul_val_annos.json',
    'test': 'COCO_mul_test_annos.json',
}

def load_split_coco(split_dir, anno_filename, default_cat_name='damage'):
    anno_path = split_dir / anno_filename
    if anno_path.exists():
        return COCOtoYOLOConverter.load_coco_json(anno_path)
    
    # If specific file is missing (e.g. unannotated test split), scan directory for images
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    images = [f for f in split_dir.iterdir() if f.suffix.lower() in img_exts] if split_dir.exists() else []
    if images:
        print(f'  Notice: {anno_filename} not found in {split_dir.name}/; indexed {len(images)} images as unannotated split.')
        return {
            'images': [{'id': i + 1, 'file_name': f.name, 'width': 0, 'height': 0} for i, f in enumerate(sorted(images))],
            'annotations': [],
            'categories': [{'id': 0, 'name': default_cat_name}]
        }
    return None

print('--- Loading Datasets ---')
for split in SPLITS:
    split_dir = RAW_DIR / split
    # 1. Damage
    cdmg = load_split_coco(split_dir, damage_anno_filenames[split], 'damage')
    if cdmg:
        coco_damage[split] = cdmg
        print(f'Damage [{split:<5}]: {len(cdmg["images"]):>2} images, {len(cdmg.get("annotations", [])):>3} annotations')
    
    # 2. Parts
    cpts = load_split_coco(split_dir, parts_anno_filenames[split], 'part')
    if cpts:
        coco_parts[split] = cpts
        print(f'Parts  [{split:<5}]: {len(cpts["images"]):>2} images, {len(cpts.get("annotations", [])):>3} annotations')

print('\n--- Damage Categories ---')
if 'train' in coco_damage:
    for cat in coco_damage['train'].get('categories', []):
        print(f"  id={cat['id']}  name={cat['name']}")

print('\n--- Parts Categories ---')
if 'train' in coco_parts:
    for cat in coco_parts['train'].get('categories', []):
        print(f"  id={cat['id']}  name={cat['name']}")

In [ ]:
# Cell 4 — Per-split and per-category annotation counts + bar chart
import matplotlib
matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print('=== Summary Table ===')
print(f"{'Split':<8} {'Damage Imgs':>12} {'Damage Anns':>12} {'Parts Imgs':>12} {'Parts Anns':>12}")
print('-' * 60)
for split in SPLITS:
    dmg_img = len(coco_damage[split]['images']) if split in coco_damage else 0
    dmg_ann = len(coco_damage[split].get('annotations', [])) if split in coco_damage else 0
    pts_img = len(coco_parts[split]['images']) if split in coco_parts else 0
    pts_ann = len(coco_parts[split].get('annotations', [])) if split in coco_parts else 0
    print(f"{split:<8} {dmg_img:>12} {dmg_ann:>12} {pts_img:>12} {pts_ann:>12}")

# Visualise Part annotations per category across splits
valid_part_splits = [s for s in ['train', 'val'] if s in coco_parts and coco_parts[s].get('annotations')]
if valid_part_splits:
    fig, axes = plt.subplots(1, len(valid_part_splits), figsize=(6 * len(valid_part_splits), 4), sharey=False)
    if len(valid_part_splits) == 1:
        axes = [axes]
    
    for ax, split in zip(axes, valid_part_splits):
        coco = coco_parts[split]
        cat_map_local = {cat['id']: cat['name'] for cat in coco.get('categories', [])}
        counts = {name: 0 for name in cat_map_local.values()}
        for ann in coco.get('annotations', []):
            cname = cat_map_local.get(ann.get('category_id', -1), 'unknown')
            counts[cname] = counts.get(cname, 0) + 1
        
        bars = ax.bar(list(counts.keys()), list(counts.values()), color='cornflowerblue', edgecolor='black', alpha=0.85)
        ax.set_title(f'Parts — {split} ({len(coco["images"])} images)', fontsize=11, fontweight='bold')
        ax.set_ylabel('Annotation Count')
        ax.tick_params(axis='x', rotation=30)
        ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        for bar in bars:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, int(yval), ha='center', va='bottom', fontsize=9)
    
    fig.suptitle('Damaged Part Category Distribution', fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    results_dir = REPO_ROOT / 'ml' / 'results' / 'detection'
    results_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(results_dir / 'annotation_counts.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved plot → {results_dir / "annotation_counts.png"}')

In [ ]:
# Cell 5 — Validate structure: file existence and bounding box integrity
from claimvision_ml.detection.coco_converter import ValidationResult

print('=== Validating Generic Damage Annotations ===')
for split, coco in coco_damage.items():
    image_dir = RAW_DIR / split
    res = COCOtoYOLOConverter.validate_structure(coco, image_dir)
    print(f'\n[Damage: {split}]')
    print(res)

print('\n=== Validating Damaged Parts Annotations ===')
for split, coco in coco_parts.items():
    image_dir = RAW_DIR / split
    res = COCOtoYOLOConverter.validate_structure(coco, image_dir)
    print(f'\n[Parts: {split}]')
    print(res)

In [ ]:
# Cell 6 — Draw original COCO boxes on sample images (BEFORE conversion)
import random

random.seed(SEED)
results_dir = REPO_ROOT / 'ml' / 'results' / 'detection'
results_dir.mkdir(parents=True, exist_ok=True)

def show_coco_samples(coco, image_dir, split, n=3, title_prefix='COCO boxes'):
    cat_map = {cat['id']: cat['name'] for cat in coco.get('categories', [])}
    ann_index = {}
    for ann in coco.get('annotations', []):
        ann_index.setdefault(ann['image_id'], []).append(ann)

    images_with_ann = [img for img in coco['images'] if img['id'] in ann_index]
    if not images_with_ann:
        return None
    sample = random.sample(images_with_ann, min(n, len(images_with_ann)))

    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, img_info in zip(axes, sample):
        fname = img_info['file_name']
        fpath = image_dir / fname
        if not fpath.exists():
            fpath = image_dir / Path(fname).name
        if not fpath.exists():
            ax.set_title(f'MISSING: {fname}')
            ax.axis('off')
            continue
        img_bgr = cv2.imread(str(fpath))
        if img_bgr is None:
            ax.set_title(f'UNREADABLE: {fname}')
            ax.axis('off')
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        anns = ann_index.get(img_info['id'], [])
        annotated = COCOtoYOLOConverter.draw_coco_boxes(img_rgb, anns, cat_map)
        ax.imshow(annotated)
        ax.set_title(f"{title_prefix}\n{Path(fname).name} ({len(anns)} boxes)", fontsize=8)
        ax.axis('off')
    fig.suptitle(f'{split} — {title_prefix}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig

for split in ['train', 'val']:
    if split in coco_parts:
        fig = show_coco_samples(coco_parts[split], RAW_DIR / split, split, n=3, title_prefix='Original COCO Parts')
        if fig:
            save_path = results_dir / f'coco_boxes_before_{split}.png'
            fig.savefig(save_path, dpi=100, bbox_inches='tight')
            plt.show()
            print(f'Saved sample view → {save_path}')

In [ ]:
# Cell 7 — Convert GENERIC DAMAGE split (nc=1)
YOLO_DAMAGE_DIR = REPO_ROOT / 'ml' / 'results' / 'detection' / 'yolo_damage'
DAMAGE_CLASS_NAMES = ['damage']

# Map all category IDs in coco_damage to class 0
DAMAGE_CAT_MAP = {}
for split, coco in coco_damage.items():
    for cat in coco.get('categories', []):
        DAMAGE_CAT_MAP[cat['id']] = 0
if not DAMAGE_CAT_MAP:
    DAMAGE_CAT_MAP = {0: 0, 1: 0}

print('Damage Category Map:', DAMAGE_CAT_MAP)
print('Damage Class Names :', DAMAGE_CLASS_NAMES)

for split, coco in coco_damage.items():
    out_split_dir = YOLO_DAMAGE_DIR / split
    image_dir = RAW_DIR / split
    COCOtoYOLOConverter.convert_split(
        coco, image_dir, out_split_dir, DAMAGE_CAT_MAP, DAMAGE_CLASS_NAMES
    )
    label_files = list((out_split_dir / 'labels').glob('*.txt'))
    image_files = list((out_split_dir / 'images').glob('*.*'))
    print(f'  {split:5s} → {len(image_files):2d} images, {len(label_files):2d} labels written')

COCOtoYOLOConverter.write_data_yaml(
    YOLO_DAMAGE_DIR,
    nc=1,
    names=DAMAGE_CLASS_NAMES,
    train_path='train/images',
    val_path='val/images',
    test_path='test/images',
)
print(f'\ndata.yaml written → {YOLO_DAMAGE_DIR / "data.yaml"}')
print('✓ Generic damage YOLO conversion complete.')

In [ ]:
# Cell 8 — Convert PART DETECTION split (nc=5)
YOLO_PARTS_DIR = REPO_ROOT / 'ml' / 'results' / 'detection' / 'yolo_parts'
PARTS_CLASS_NAMES = ['headlamp', 'front_bumper', 'hood', 'door', 'rear_bumper']

_name_to_yolo = {
    'headlamp': 0,
    'front_bumper': 1, 'front bumper': 1,
    'hood': 2,
    'door': 3,
    'rear_bumper': 4, 'rear bumper': 4,
}

PARTS_CAT_MAP = {}
for split, coco in coco_parts.items():
    for cat in coco.get('categories', []):
        raw_name = cat['name'].lower().strip()
        norm_name = raw_name.replace(' ', '_')
        if norm_name in _name_to_yolo:
            PARTS_CAT_MAP[cat['id']] = _name_to_yolo[norm_name]
        elif raw_name in _name_to_yolo:
            PARTS_CAT_MAP[cat['id']] = _name_to_yolo[raw_name]
        elif cat['id'] in [0, 1, 2, 3, 4]:
            PARTS_CAT_MAP[cat['id']] = cat['id']

print('Parts Category Map:', PARTS_CAT_MAP)
print('Parts Class Names :', PARTS_CLASS_NAMES)

for split, coco in coco_parts.items():
    out_split_dir = YOLO_PARTS_DIR / split
    image_dir = RAW_DIR / split
    COCOtoYOLOConverter.convert_split(
        coco, image_dir, out_split_dir, PARTS_CAT_MAP, PARTS_CLASS_NAMES
    )
    label_files = list((out_split_dir / 'labels').glob('*.txt'))
    image_files = list((out_split_dir / 'images').glob('*.*'))
    print(f'  {split:5s} → {len(image_files):2d} images, {len(label_files):2d} labels written')

COCOtoYOLOConverter.write_data_yaml(
    YOLO_PARTS_DIR,
    nc=5,
    names=PARTS_CLASS_NAMES,
    train_path='train/images',
    val_path='val/images',
    test_path='test/images',
)
print(f'\ndata.yaml written → {YOLO_PARTS_DIR / "data.yaml"}')
print('✓ Damaged parts YOLO conversion complete.')

In [ ]:
# Cell 9 — Draw YOLO boxes on same sample images (AFTER conversion) — side-by-side
def read_yolo_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id, cx, cy, nw, nh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x = (cx - nw / 2) * img_w
        y = (cy - nh / 2) * img_h
        w = nw * img_w
        h = nh * img_h
        boxes.append({'bbox': [x, y, w, h], 'category_id': cls_id})
    return boxes

for split in ['train', 'val']:
    if split not in coco_parts:
        continue
    coco = coco_parts[split]
    image_dir = RAW_DIR / split
    ann_index = {}
    for ann in coco.get('annotations', []):
        ann_index.setdefault(ann['image_id'], []).append(ann)

    images_with_ann = [img for img in coco['images'] if img['id'] in ann_index]
    if not images_with_ann:
        continue
    random.seed(SEED)
    sample = random.sample(images_with_ann, min(3, len(images_with_ann)))

    cat_map_damage = {0: 'damage'}
    cat_map_parts = {i: name for i, name in enumerate(PARTS_CLASS_NAMES)}

    fig, axes = plt.subplots(len(sample), 2, figsize=(10, 4 * len(sample)))
    if len(sample) == 1:
        axes = [axes]

    for row_idx, img_info in enumerate(sample):
        fname = Path(img_info['file_name']).name
        fpath = image_dir / fname
        if not fpath.exists():
            continue
        img_bgr = cv2.imread(str(fpath))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_bgr.shape[:2]
        stem = Path(fname).stem

        # 1. YOLO Damage
        dmg_txt = YOLO_DAMAGE_DIR / split / 'labels' / f'{stem}.txt'
        dmg_boxes = read_yolo_boxes(dmg_txt, w, h)
        dmg_vis = COCOtoYOLOConverter.draw_coco_boxes(img_rgb, dmg_boxes, cat_map_damage)
        axes[row_idx][0].imshow(dmg_vis)
        axes[row_idx][0].set_title(f'YOLO Damage — {fname} ({len(dmg_boxes)} boxes)', fontsize=8)
        axes[row_idx][0].axis('off')

        # 2. YOLO Parts
        pts_txt = YOLO_PARTS_DIR / split / 'labels' / f'{stem}.txt'
        pts_boxes = read_yolo_boxes(pts_txt, w, h)
        pts_vis = COCOtoYOLOConverter.draw_coco_boxes(img_rgb, pts_boxes, cat_map_parts)
        axes[row_idx][1].imshow(pts_vis)
        axes[row_idx][1].set_title(f'YOLO Parts — {fname} ({len(pts_boxes)} boxes)', fontsize=8)
        axes[row_idx][1].axis('off')

    fig.suptitle(f'{split} — YOLO Conversion Verification (Left: Damage, Right: Parts)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    save_path = results_dir / f'yolo_boxes_after_{split}.png'
    fig.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved verified boxes view → {save_path}')

In [ ]:
# Cell 10 — Run conversion assertions for both tasks
print('=== Running Conversion Assertions ===\n')

print('--- Generic Damage Task ---')
for split in SPLITS:
    split_dir = YOLO_DAMAGE_DIR / split
    if split_dir.exists():
        print(f'[{split}]', end=' ')
        COCOtoYOLOConverter.run_conversion_assertions(split_dir)

print('\n--- Damaged Parts Task ---')
for split in SPLITS:
    split_dir = YOLO_PARTS_DIR / split
    if split_dir.exists():
        print(f'[{split}]', end=' ')
        COCOtoYOLOConverter.run_conversion_assertions(split_dir)

In [ ]:
# Cell 11 — Print data.yaml contents for both tasks
import yaml

for task_name, yolo_dir in [('Generic Damage', YOLO_DAMAGE_DIR), ('Part Detection', YOLO_PARTS_DIR)]:
    yaml_file = yolo_dir / 'data.yaml'
    print(f'=== {task_name} Configuration: {yaml_file} ===')
    if yaml_file.exists():
        data = yaml.safe_load(yaml_file.read_text(encoding='utf-8'))
        print(yaml.dump(data, default_flow_style=False))
    print()

In [ ]:
# Cell 12 — Print YOLO directory trees with label and image counts
def print_dir_summary(base_dir):
    base = Path(base_dir)
    print(f'{base.name}/')
    for split in ['train', 'val', 'test']:
        s_dir = base / split
        if s_dir.exists():
            imgs = len(list((s_dir / 'images').glob('*.*'))) if (s_dir / 'images').exists() else 0
            lbls = len(list((s_dir / 'labels').glob('*.txt'))) if (s_dir / 'labels').exists() else 0
            print(f'  ├── {split}/ ({imgs} images, {lbls} label files)')
    if (base / 'data.yaml').exists():
        print('  └── data.yaml')

print('=== YOLO Damage Directory Summary ===')
print_dir_summary(YOLO_DAMAGE_DIR)
print('\n=== YOLO Parts Directory Summary ===')
print_dir_summary(YOLO_PARTS_DIR)

In [ ]:
# Cell 13 — Reproducibility: versions, seed, checksums
import hashlib
import platform

print('=== Reproducibility Record ===')
print(f'Python      : {sys.version}')
print(f'Platform    : {platform.platform()}')
print(f'SEED        : {SEED}')
print(f'Task ID     : DET-COCO-001')
print()

for task_name, yolo_dir in [('yolo_damage', YOLO_DAMAGE_DIR), ('yolo_parts', YOLO_PARTS_DIR)]:
    yaml_file = yolo_dir / 'data.yaml'
    if yaml_file.exists():
        sha = hashlib.sha256(yaml_file.read_bytes()).hexdigest()[:16]
        print(f'{task_name}/data.yaml SHA-256 (first 16): {sha}')

## Cell 14 — Findings and Limitations

### Dataset and Conversion Summary

- **Generic Damage Task** (`yolo_damage/`):
  - Formatted for single-class damage localisation (`nc=1`, name: `damage`).
  - All images across `train`, `val`, and `test` have matching `.txt` label files.
- **Damaged Parts Task** (`yolo_parts/`):
  - Formatted for 5-part localisation (`nc=5`, names: `headlamp`, `front_bumper`, `hood`, `door`, `rear_bumper`).
  - Bounding box coordinates converted from COCO `[x, y, w, h]` to YOLO normalised `[cx, cy, nw, nh]`.
- **Validation & Round-Trip Assertions**:
  - All bounding boxes fall within image boundaries.
  - Round-trip floating point precision verified within `1e-6`.

### Next Steps

→ **Notebook 11** (`11_yolo_damage_training.ipynb`) — Train YOLOv8n generic damage detector using `yolo_damage/data.yaml`.
→ **Notebook 12** (`12_yolo_part_training.ipynb`) — Train YOLOv8n damaged parts detector using `yolo_parts/data.yaml`.